# Aula 7 - Operações Morfológicas

Nesta aula, exploraremos as principais operações morfológicas em imagens binárias e em tons de cinza: Dilatação, Erosão, Abertura, Fechamento, Gradiente Morfológico, Top Hat e Black Hat.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Configuração de caminhos
INPUT_DIR = Path("data/input")
OUTPUT_DIR = Path("data/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def show_image(title, img, cmap='gray'):
    plt.figure(figsize=(8, 6))
    if len(img.shape) == 3:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis('off')
    plt.show()

## Experimento 1.1
Carregue a imagem `rice.png`, binarize e aplique operações morfológicas de dilatação, erosão, abertura e fechamento. Mostre e compare os resultados.

In [ ]:
# Carregar a imagem rice.png
rice_img = cv2.imread(str(INPUT_DIR / "rice.png"), cv2.IMREAD_GRAYSCALE)

# Binarização com limiar de Otsu (devido à iluminação não uniforme)
_, thresh_rice = cv2.threshold(rice_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Definir elemento estruturante retangular 3x3
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))

# Aplicar operações morfológicas
dilated = cv2.dilate(thresh_rice, kernel, iterations=1)
eroded = cv2.erode(thresh_rice, kernel, iterations=1)
opened = cv2.morphologyEx(thresh_rice, cv2.MORPH_OPEN, kernel)
closed = cv2.morphologyEx(thresh_rice, cv2.MORPH_CLOSE, kernel)

# Exibição dos resultados
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes[0, 0].imshow(rice_img, cmap='gray'); axes[0, 0].set_title('Original'); axes[0, 0].axis('off')
axes[0, 1].imshow(thresh_rice, cmap='gray'); axes[0, 1].set_title('Binarizada (Otsu)'); axes[0, 1].axis('off')
axes[0, 2].imshow(dilated, cmap='gray'); axes[0, 2].set_title('Dilatação'); axes[0, 2].axis('off')
axes[1, 0].imshow(eroded, cmap='gray'); axes[1, 0].set_title('Erosão'); axes[1, 0].axis('off')
axes[1, 1].imshow(opened, cmap='gray'); axes[1, 1].set_title('Abertura'); axes[1, 1].axis('off')
axes[1, 2].imshow(closed, cmap='gray'); axes[1, 2].set_title('Fechamento'); axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

# Salvar resultados
cv2.imwrite(str(OUTPUT_DIR / "exp1_1_rice_binarized.png"), thresh_rice)
cv2.imwrite(str(OUTPUT_DIR / "exp1_1_rice_dilated.png"), dilated)
cv2.imwrite(str(OUTPUT_DIR / "exp1_1_rice_eroded.png"), eroded)
cv2.imwrite(str(OUTPUT_DIR / "exp1_1_rice_opened.png"), opened)
cv2.imwrite(str(OUTPUT_DIR / "exp1_1_rice_closed.png"), closed)

**Respostas:**
- Qual o efeito observado ao aplicar a **Abertura** no arroz binarizado? (Preencha aqui)
- Qual o efeito observado ao aplicar o **Fechamento**? (Preencha aqui)

## Experimento 1.2
Carregue a imagem `placa.png`, binarize e aplique operações morfológicas de dilatação e erosão. Compare os resultados. O objetivo é melhorar a identificação dos números da placa.

In [ ]:
# Carregar a imagem placa.png
placa_img = cv2.imread(str(INPUT_DIR / "placa.png"), cv2.IMREAD_GRAYSCALE)

# Binarização com limiar de Otsu
_, thresh_placa = cv2.threshold(placa_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Definir elemento estruturante retangular 3x3
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))

# Aplicar dilatação e erosão
dilated_placa = cv2.dilate(thresh_placa, kernel, iterations=1)
eroded_placa = cv2.erode(thresh_placa, kernel, iterations=1)

# Exibição
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(thresh_placa, cmap='gray'); axes[0].set_title('Binarizada (Original)'); axes[0].axis('off')
axes[1].imshow(dilated_placa, cmap='gray'); axes[1].set_title('Dilatação'); axes[1].axis('off')
axes[2].imshow(eroded_placa, cmap='gray'); axes[2].set_title('Erosão'); axes[2].axis('off')

plt.tight_layout()
plt.show()

# Salvar resultados
cv2.imwrite(str(OUTPUT_DIR / "exp1_2_placa_bin.png"), thresh_placa)
cv2.imwrite(str(OUTPUT_DIR / "exp1_2_placa_dilated.png"), dilated_placa)
cv2.imwrite(str(OUTPUT_DIR / "exp1_2_placa_eroded.png"), eroded_placa)

**Respostas:**
- Qual das operações (dilatação ou erosão) melhorou a identificação dos caracteres da placa? Por quê? (Preencha aqui considerando se os caracteres são pixels pretos ou brancos na binarização)

## Experimento 1.3
Carregue a imagem `moedas.png`, binarize e aplique operações morfológicas (dilatação, erosão, abertura, fechamento), aritméticas e lógicas (caso necessário). O objetivo é identificar as bordas de cada moeda.

In [ ]:
# Carregar moedas.png
moedas_img = cv2.imread(str(INPUT_DIR / "moedas.png"), cv2.IMREAD_GRAYSCALE)

# Binarização com Otsu (invertida pois o fundo é mais claro e queremos moedas brancas no fundo preto)
_, thresh_moedas = cv2.threshold(moedas_img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

# Preenchimento de furos usando fechamento com um elemento estruturante circular
kernel_circular = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
closed_moedas = cv2.morphologyEx(thresh_moedas, cv2.MORPH_CLOSE, kernel_circular)

# Obter borda por Gradiente Morfológico
border = cv2.morphologyEx(closed_moedas, cv2.MORPH_GRADIENT, cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)))

# Exibição
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(moedas_img, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(thresh_moedas, cmap='gray'); axes[1].set_title('Binarizada (Invertida)'); axes[1].axis('off')
axes[2].imshow(closed_moedas, cmap='gray'); axes[2].set_title('Fechamento (Sólido)'); axes[2].axis('off')
axes[3].imshow(border, cmap='gray'); axes[3].set_title('Bordas (Gradiente)'); axes[3].axis('off')

plt.tight_layout()
plt.show()

# Salvar resultados
cv2.imwrite(str(OUTPUT_DIR / "exp1_3_moedas_bin.png"), thresh_moedas)
cv2.imwrite(str(OUTPUT_DIR / "exp1_3_moedas_closed.png"), closed_moedas)
cv2.imwrite(str(OUTPUT_DIR / "exp1_3_moedas_border.png"), border)

**Respostas:**
- Qual a importância de aplicar uma operação de fechamento antes de extrair os contornos das moedas? (Preencha aqui)
- Explique matematicamente como o gradiente morfológico extrai as bordas de um objeto. (Preencha aqui)

## Experimento 2
Aplique as operações de dilatação, erosão, abertura e fechamento na imagem `RuidoBinario.png` utilizando diferentes tamanhos de elementos estruturantes.

In [ ]:
# Carregar imagem RuidoBinario.png
ruido_img = cv2.imread(str(INPUT_DIR / "RuidoBinario.png"), cv2.IMREAD_GRAYSCALE)

# Definir tamanhos de kernel para testar
sizes = [3, 5, 9]

for size in sizes:
    kernel_rect = cv2.getStructuringElement(cv2.MORPH_RECT, (size, size))
    
    dil = cv2.dilate(ruido_img, kernel_rect)
    ero = cv2.erode(ruido_img, kernel_rect)
    ope = cv2.morphologyEx(ruido_img, cv2.MORPH_OPEN, kernel_rect)
    clo = cv2.morphologyEx(ruido_img, cv2.MORPH_CLOSE, kernel_rect)
    
    # Exibir para o tamanho atual de kernel
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    fig.suptitle(f"Kernel Retangular {size}x{size}", fontsize=14)
    axes[0].imshow(ruido_img, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(dil, cmap='gray'); axes[1].set_title('Dilatação'); axes[1].axis('off')
    axes[2].imshow(ero, cmap='gray'); axes[2].set_title('Erosão'); axes[2].axis('off')
    axes[3].imshow(ope, cmap='gray'); axes[3].set_title('Abertura'); axes[3].axis('off')
    axes[4].imshow(clo, cmap='gray'); axes[4].set_title('Fechamento'); axes[4].axis('off')
    plt.show()
    
    # Salvar resultados
    cv2.imwrite(str(OUTPUT_DIR / f"exp2_dilated_k{size}.png"), dil)
    cv2.imwrite(str(OUTPUT_DIR / f"exp2_eroded_k{size}.png"), ero)
    cv2.imwrite(str(OUTPUT_DIR / f"exp2_opened_k{size}.png"), ope)
    cv2.imwrite(str(OUTPUT_DIR / f"exp2_closed_k{size}.png"), clo)

**Respostas:**
- Qual o impacto do aumento do tamanho do elemento estruturante na **Abertura** e no **Fechamento** desta imagem com ruído? (Preencha aqui)
- Qual operação foi mais eficaz para remover pontos brancos no fundo? E para tapar furos pretos nos objetos? (Preencha aqui)

## Experimento 3
Aplique operações morfológicas na imagem `Morfo2ComRuido.png` para remover ruídos internos (furos pretos) e externos (pontos brancos), gerando o resultado esperado.

In [ ]:
# Carregar Morfo2ComRuido.png
morfo2_img = cv2.imread(str(INPUT_DIR / "Morfo2ComRuido.png"), cv2.IMREAD_GRAYSCALE)

# Kernel retangular 5x5 para filtrar o ruído maior
kernel_5x5 = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))

# Aplicação de Abertura (para remover ruídos brancos externos) seguida de Fechamento (para remover furos pretos internos)
opened_then_closed = cv2.morphologyEx(
    cv2.morphologyEx(morfo2_img, cv2.MORPH_OPEN, kernel_5x5),
    cv2.MORPH_CLOSE,
    kernel_5x5
)

# Aplicação de Fechamento seguida de Abertura (ordem inversa para comparação)
closed_then_opened = cv2.morphologyEx(
    cv2.morphologyEx(morfo2_img, cv2.MORPH_CLOSE, kernel_5x5),
    cv2.MORPH_OPEN,
    kernel_5x5
)

# Exibição
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(morfo2_img, cmap='gray'); axes[0].set_title('Original com Ruído'); axes[0].axis('off')
axes[1].imshow(opened_then_closed, cmap='gray'); axes[1].set_title('Abertura -> Fechamento'); axes[1].axis('off')
axes[2].imshow(closed_then_opened, cmap='gray'); axes[2].set_title('Fechamento -> Abertura'); axes[2].axis('off')

plt.tight_layout()
plt.show()

# Salvar melhor resultado
cv2.imwrite(str(OUTPUT_DIR / "exp3_cleaned_op_cl.png"), opened_then_closed)
cv2.imwrite(str(OUTPUT_DIR / "exp3_cleaned_cl_op.png"), closed_then_opened)

**Respostas:**
- A ordem em que a abertura e o fechamento são executados alterou o resultado final? Explique a diferença visual e lógica. (Preencha aqui)

## Experimento 4
Implemente a detecção de contorno na imagem `contorno_original.png` utilizando a operação de Gradiente Morfológico (`MORPH_GRADIENT`).

In [ ]:
# Carregar a imagem contorno_original.png
contorno_img = cv2.imread(str(INPUT_DIR / "contorno_original.png"), cv2.IMREAD_GRAYSCALE)

# Binarização simples
_, thresh_contorno = cv2.threshold(contorno_img, 127, 255, cv2.THRESH_BINARY)

# Kernel retangular 3x3 para contorno fino
kernel_3x3 = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))

# Gradiente morfológico (Diferença entre dilatação e erosão)
gradient_contorno = cv2.morphologyEx(thresh_contorno, cv2.MORPH_GRADIENT, kernel_3x3)

# Exibição
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(contorno_img, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(gradient_contorno, cmap='gray'); axes[1].set_title('Contornos (Gradiente Morfológico)'); axes[1].axis('off')

plt.tight_layout()
plt.show()

# Salvar resultado
cv2.imwrite(str(OUTPUT_DIR / "exp4_gradient_contour.png"), gradient_contorno)

**Respostas:**
- Qual a diferença entre as bordas detectadas com o Gradiente Morfológico em comparação com um detector clássico como Sobel ou Canny? (Preencha aqui)

## Experimento 5
Utilize a imagem `lena.png` em escala de cinza e experimente outras operações morfológicas disponíveis no OpenCV: Gradiente Morfológico, Top Hat e Black Hat.

In [ ]:
# Carregar lena.png em escala de cinza
lena_img = cv2.imread(str(INPUT_DIR / "lena.png"), cv2.IMREAD_GRAYSCALE)

# Elemento estruturante retangular 9x9 para evidenciar os efeitos em escala de cinza
kernel_9x9 = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))

# Operações morfológicas em escala de cinza
grad_gray = cv2.morphologyEx(lena_img, cv2.MORPH_GRADIENT, kernel_9x9)
tophat_gray = cv2.morphologyEx(lena_img, cv2.MORPH_TOPHAT, kernel_9x9)
blackhat_gray = cv2.morphologyEx(lena_img, cv2.MORPH_BLACKHAT, kernel_9x9)

# Exibição
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes[0, 0].imshow(lena_img, cmap='gray'); axes[0, 0].set_title('Original (Tons de Cinza)'); axes[0, 0].axis('off')
axes[0, 1].imshow(grad_gray, cmap='gray'); axes[0, 1].set_title('Gradiente Morfológico (Bordas)'); axes[0, 1].axis('off')
axes[1, 0].imshow(tophat_gray, cmap='gray'); axes[1, 0].set_title('Top Hat (Elementos Claros < Kernel)'); axes[1, 0].axis('off')
axes[1, 1].imshow(blackhat_gray, cmap='gray'); axes[1, 1].set_title('Black Hat (Elementos Escuros < Kernel)'); axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

# Salvar resultados
cv2.imwrite(str(OUTPUT_DIR / "exp5_lena_gradient.png"), grad_gray)
cv2.imwrite(str(OUTPUT_DIR / "exp5_lena_tophat.png"), tophat_gray)
cv2.imwrite(str(OUTPUT_DIR / "exp5_lena_blackhat.png"), blackhat_gray)

**Respostas:**
- O que as operações **Top Hat** e **Black Hat** evidenciam na imagem em tons de cinza? (Preencha aqui)
- Dê um exemplo prático de aplicação para a operação Top Hat. (Preencha aqui)